In [ ]:
#import packages and fech api keys from env file
import os
import sys
sys.path.append("..")
from util import check_api_key
from dotenv import load_dotenv
from openai import OpenAI
import ollama
import anthropic
import gradio as gr 
from scraper import fetch_website_links, fetch_website_contents
from IPython.display import Markdown, display, update_display
import json

openai_api_key = check_api_key("openai")
anthropic_api_key = check_api_key("anthropic")



In [ ]:
#connect to openai and anthropic
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)


In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [ ]:
system_message = "You are a helpful assistant who responds in simple and easy to understand sentences. " \
"You respond in markdown without code blocks."

def stream_gpt(prompt):
    messages = [
        {"role" : "system", "content": brochure_system_prompt},
        {"role" : "user", "content": prompt}
    ]
    stream = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages,
        stream = True
    )

    result = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        update_display(Markdown(result), display_id=display_handle.display_id)
        #yield result


def stream_claude(prompt):
    messages = [
        {"role" : "system", "content" : brochure_system_prompt},
        {"role" : "user", "content" : prompt}
    ]

    stream = anthropic.chat.completions.create(
        model  = "claude-haiku-4-5",
        messgaes = messages,
        stream = True
    )

    result = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        update_display(Markdown(result), display_id=display_handle.display_id)
        #yield result


In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):

        """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

def slect_relevant_links(url, model):
    response = openai.chat.completions.create(
        model = model,
        messages = [
            {"role" : "system" , "content" : link_system_prompt},
            {"role" : "user" , "content" : get_links_user_prompt(url)}
        ],
        response_format = {"type" : "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    #print(f"Found {len(links['links'])} relevant links")
    return links

def fetch_page_and_relevant_links(url, model):
    contents = fetch_website_contents(url)
    relevant_links = slect_relevant_links(url, model)

    result  = f"## Landing page:\n\n {contents} \n\n ## Relevnt Links: \n"
    for link in relevant_links['links']:
        result += f"\n Link: {link['type']}\n"
        result += fetch_website_contents(link['url'])
    return result

In [ ]:

def create_brochure_user_prompt(company_name, url, model):
    user_prompt = f"""You are looking at a company called {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt = fetch_page_and_relevant_links(url, model)
    user_prompt = user_prompt[:5000]
    return user_prompt

def stream_brochure(company_name, url, model):
    gpt_model = "gpt-5-mini"
    claude_model = "claude-haiku-4-5"
    yield ""

    if model == "GPT":
        user_prompt = create_brochure_user_prompt(company_name, url, gpt_model)
        result = stream_gpt(user_prompt)
    elif model == "Claude":
        user_prompt = create_brochure_user_prompt(company_name, url, claude_model)
        result = stream_claude(user_prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
name_input = gr.Textbox(label="Company name: ")
url_input = gr.Textbox(label = "Landing page url including http:// or https://")
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response: ");

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[name_input, url_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Hugging Face", "https://huggingface.co", "GPT"],
        ["Edward Donner", "https://edwarddonner.com", "Claude"]
    ],
    flagging_mode="never"
)
view.launch()